# 섹션3-8. AI를 활용한 2D Density Plot, Hexabin Plot

> 강의: [32가지 데이터 시각화 전략 - 비전공자를 위한 기초이론 & 실습](https://www.inflearn.com/course/32-data-visualizatio/dashboard?cid=343563) (반병현) — 전체 19강

- [x] 강의 시청 완료
- [x] 실습/정리 완료

## 배운 내용

<!-- 강의를 보면서 핵심을 적는다 -->

-

## 목표 / 재현할 것

<!-- 이 강의에서 만든 차트를 내 방식대로 다시 만들어본다 -->

-


## 실습

> 강의는 3군집 + ABC 그룹 + 이상치가 섞인 실습용 더미데이터를 썼다. 그 파일은
> `.gitignore` 처리(재배포 권한 불명확)했으니, 같은 구조(군집 3개, X·Y 2변수)를
> `viz_utils.load_sample("iris_like")`로 재현해서 진행한다.

In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_utils import setup, load_sample

setup()
rng = np.random.default_rng(0)

df = load_sample("iris_like", seed=1)
df = df.rename(columns={"x1": "X", "x2": "Y"})[["X", "Y", "군집"]]
print(f"{len(df):,}행, 군집 {df['군집'].unique().tolist()}")
df.head()

### 1. 그냥 산점도 — 점이 너무 많으면

In [ ]:
# 표본을 늘려서 강의가 지적한 "너무 많이 찍힌" 상태를 실제로 만든다
big = pd.concat([load_sample("iris_like", seed=s) for s in range(1, 21)], ignore_index=True)
big = big.rename(columns={"x1": "X", "x2": "Y"})

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(big["X"], big["Y"], s=6, alpha=0.4, color="#4C78A8")
ax.set_title(f"산점도 (n={len(big):,}) — 가운데가 뭉개진다")
plt.show()

> "점이 너무 많이 찍혀있죠 가운데. 이게 얼마나 많은 점들이 몰려 있는 건지 감이
> 잘 안 옵니다." — 강의 그대로다. 밀도로 바꿔본다.

### 2. 2D 밀도(등고선) — 얼마나 몰려 있는지

In [ ]:
from scipy.stats import gaussian_kde

xy = np.vstack([big["X"], big["Y"]])
kde = gaussian_kde(xy)

xg, yg = np.mgrid[big["X"].min():big["X"].max():100j, big["Y"].min():big["Y"].max():100j]
zg = kde(np.vstack([xg.ravel(), yg.ravel()])).reshape(xg.shape)

fig, ax = plt.subplots(figsize=(5.5, 5))
cs = ax.contourf(xg, yg, zg, levels=12, cmap="YlOrRd")
fig.colorbar(cs, ax=ax, label="밀도")
ax.set_title("2D 밀도 (등고선) — 진할수록 점이 몰려 있다")
plt.show()

### 3. 헥스빈 — 육각형이어야 하는 이유

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].hist2d(big["X"], big["Y"], bins=25, cmap="YlOrRd")
axes[0].set_title("사각형 bin (2D 히스토그램)")

hb = axes[1].hexbin(big["X"], big["Y"], gridsize=22, cmap="YlOrRd")
axes[1].set_title("육각형 bin (진짜 헥스빈)")

plt.tight_layout()
plt.show()

> 강의에서 AI가 처음 만든 "헥사빈"이 실은 **사각형**이었고, 강사님이 "헥사빈은
> 육각형이어야 한다"고 다시 요청한 장면이 나온다. 이유가 있다 — 사각형은 격자 경계가
> 인접한 두 방향(가로·세로)으로만 붙는데, **육각형은 6방향으로 이웃과 맞닿아 경계가
> 더 매끄럽고 뭉침을 덜 과장한다.** 같은 데이터·같은 격자 수로 그려도 사각형 쪽이
> 블록처럼 뚝뚝 끊겨 보이는 걸 위 그림에서 확인할 수 있다.

### 4. 해상도(bin 개수)를 바꾸면

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, n in zip(axes, [8, 22, 45]):
    ax.hexbin(big["X"], big["Y"], gridsize=n, cmap="YlOrRd")
    ax.set_title(f"gridsize={n}")
fig.suptitle("해상도를 높이면 쪼글쪼글해지고, 낮추면 뭉뚱그려진다 — 강의에서 본 그대로")
plt.tight_layout()
plt.show()

---

## 메모

-
